# Setup

In [ ]:
import os
import psycopg
import pandas as pd
from sqlalchemy import create_engine, text

In [ ]:
# pd.set_option(
#     # "display.float_format",
#     # lambda x: f"{x:.2e}" if abs(x) < 0.01 and x != 0 else f"{x:.2f}",
#     "display.max_rows",
#     100,
#     "display.max_columns",
#     100,
# )

# ETL

In [ ]:
!psql $DATABASE_URL -f ../sql/ddl.sql
!psql $DATABASE_URL -f ../sql/load.sql
!psql $DATABASE_URL -f ../sql/transform.sql

CREATE TABLE
CREATE TABLE
CREATE TABLE
CREATE TABLE
TRUNCATE TABLE
TRUNCATE TABLE
TRUNCATE TABLE
TRUNCATE TABLE
COPY 2018352
COPY 41919
COPY 637
COPY 15286
psql:../sql/transform.sql:1: NOTICE:  table "merged" does not exist, skipping
DROP TABLE
SELECT 2018352


# Queries

In [ ]:
db_url = os.environ["DATABASE_URL"]
engine = create_engine(
    db_url.replace("postgresql://", "postgresql+psycopg://")
)

In [ ]:
pd.read_sql(
    """
SELECT *
FROM merged
ORDER BY
    datetime,
    county,
    product_type,
    is_business,
    is_consumption,
    target 
LIMIT 5;
""",
    engine,
)

,datetime,county,product_type,is_business,is_consumption,target,data_block_id,prediction_unit_id,eic_count,installed_capacity,lowest_price_per_mwh,highest_price_per_mwh,euros_per_mwh,row_id
0,2021-09-01,0,0,True,False,0.000,0,3,None,None,None,None,None,6
1,2021-09-01,0,0,True,True,59.000,0,3,None,None,None,None,None,7
2,2021-09-01,0,1,False,False,0.713,0,0,None,None,None,None,None,0
3,2021-09-01,0,1,False,True,96.590,0,0,None,None,None,None,None,1
4,2021-09-01,0,1,True,False,0.000,0,4,None,None,None,None,None,8


## Nulls

In [ ]:
pd.read_sql(
    """
SELECT
    COUNT(*) FILTER (WHERE target IS NULL) AS target_nulls,
    COUNT(*) FILTER (WHERE datetime IS NULL) AS datetime_nulls,
    COUNT(*) FILTER (WHERE eic_count IS NULL) AS eic_nulls,
    COUNT(*) FILTER (WHERE installed_capacity IS NULL) AS installed_capacity_nulls,
    COUNT(*) FILTER (WHERE lowest_price_per_mwh IS NULL) AS lowest_price_per_mwh_nulls,
    COUNT(*) FILTER (WHERE highest_price_per_mwh IS NULL) AS highest_price_per_mwh_nulls,
    COUNT(*) FILTER (WHERE euros_per_mwh IS NULL) AS euros_per_mwh_nulls
FROM merged;
""",
    engine,
)

,target_nulls,datetime_nulls,eic_nulls,installed_capacity_nulls,lowest_price_per_mwh_nulls,highest_price_per_mwh_nulls,euros_per_mwh_nulls
0,528,0,8640,8640,2928,2928,3196


- `target_nulls` - missing values that appear because of the shift from/to daylight saving time.
- `eic_nulls`, `installed_capacity_nulls`, `lowest_price_per_mwh_nulls`, `highest_price_per_mwh_nulls`, `euros_per_mwh_nulls` - missing values for the first days of the dataset, because related data is available only with a lag.

## Time Series Completeness

In [ ]:
pd.read_sql(
    """
SELECT
    COUNT(*) AS number_of_timeseries,
    MIN(number_of_entries) AS min_entries,
    ROUND(AVG(number_of_entries), 2) AS avg_entries,
    PERCENTILE_DISC(0.5) WITHIN GROUP (ORDER BY number_of_entries) AS median_entries,
    MAX(number_of_entries) AS max_entries
FROM (
    SELECT
        county,
        product_type,
        is_business,
        is_consumption,
        COUNT(target) AS number_of_entries
    FROM merged
    GROUP BY county, product_type, is_business, is_consumption
) AS timeseries_completeness;
""",
    engine,
)

,number_of_timeseries,min_entries,avg_entries,median_entries,max_entries
0,138,1656,14621.91,15308,15308


There are 138 unique time series, most of them are complete (median = max = 15308). A few time series have significantly fewer entries (min = 1656), which is explored and visualized in "`1. Exploratory Data Analysis.ipynb`".

## Average Consumption and Production

In [ ]:
pd.read_sql(
    """
SELECT
    is_consumption,
    ROUND(AVG(sum_target)::numeric, 2) / 1_000_000 AS "avg_monthly_target_GWh"
FROM (
    SELECT
        DATE_TRUNC('month', datetime) as month,
        is_consumption,
        SUM(target) AS sum_target
    FROM merged
    GROUP BY DATE_TRUNC('month', datetime), is_consumption
) AS sum_target_month
GROUP BY is_consumption;
""",
    engine,
)

,is_consumption,avg_monthly_target_GWh
0,False,4.276011
1,True,22.133996


Monthly consumption ~5 times greater than monthly production (22.1 vs 4.3 GWh avg).

## Installed capacity CAGR 

In [ ]:
pd.read_sql(
    """
WITH bounds AS (
    SELECT MIN(datetime) AS first_dt, MAX(datetime) AS last_dt
    FROM merged
    WHERE installed_capacity IS NOT NULL
),
vals as (
    SELECT
        SUM(CASE WHEN datetime = first_dt THEN installed_capacity END) AS first_val,
        SUM(CASE WHEN datetime = last_dt THEN installed_capacity END) AS last_val
    FROM merged, bounds
    WHERE installed_capacity IS NOT NULL
)

SELECT ROUND(((
    POWER(
        last_val / first_val, 
        1.0 / (EXTRACT(EPOCH FROM (last_dt - first_dt)) / 86400 / 365.25)
    ) - 1) * 100)::numeric,
    2
) AS "capacity_CAGR"
FROM bounds, vals
""",
    engine,
)

,capacity_CAGR
0,53.03


Installed capacity CAGR is 53%, reflecting rapid growth of prosumer solar installations in Estonia during 2021-2023.

## Consumption and Production by Categories

In [ ]:
category_list = ["is_business", "product_type", "county"]

for category in category_list:
    display(
        pd.read_sql(
            f"""
SELECT
    {category},
    is_consumption,
    ROUND(AVG(target)::numeric, 2) AS avg_hourly_target,
    SUM(target) AS total_target
FROM merged
GROUP by is_consumption, {category} 
ORDER BY is_consumption, avg_hourly_target DESC 
""",
            engine,
        )
    )

,is_business,is_consumption,avg_hourly_target,total_target
0,False,False,98.96,4.624386e+07
1,True,False,80.41,4.355238e+07
2,True,True,744.64,4.033063e+08
3,False,True,131.62,6.150759e+07


,product_type,is_consumption,avg_hourly_target,total_target
0,3,False,152.74,7.014522e+07
1,1,False,41.12,1.606438e+07
2,0,False,37.29,3.178918e+06
3,2,False,5.53,4.077131e+05
4,3,True,809.28,3.716535e+08
5,0,True,438.76,3.740390e+07
6,1,True,136.35,5.327455e+07
7,2,True,33.67,2.482004e+06


,county,is_consumption,avg_hourly_target,total_target
0,0,False,256.19,2.726804e+07
1,11,False,128.20,1.268863e+07
2,14,False,99.47,6.255745e+06
3,7,False,90.08,7.793578e+06
4,10,False,82.64,5.562032e+06
5,5,False,82.05,6.218842e+06
6,3,False,61.19,3.746996e+06
7,13,False,57.94,3.505779e+06
8,9,False,56.70,3.471846e+06
9,8,False,55.55,2.551259e+06


- Business segment is consuming far more than non-business (745 vs 132 avg), which is explained by industrial facilities. For production the difference is much smaller: non-business prosumers produce at a comparable level (99 vs 80 avg).
- Product type 3 (spot contract) dominates in both categories (809 and 153 avg for consumption and production). Product type 2 (general service) has the smallest volumes of both production (6 avg) and consumption (34 avg).
- County 0 (Harjumaa) differs significantly from second place (Tartumaa) for both consumption (1626 vs 915) and production (256 vs 128 avg). County 12 (unknown, possibly intercounty or aggregated data) has the least avg production (6.5) but third place for consumption (547 avg). The least avg consumption is in county 1 (Hiiumaa, the smallest county in Estonia) - 48.

## Month-Over-Month Consumption Change

In [ ]:
pd.read_sql(
    """
WITH monthly_cons AS (
    SELECT DATE_TRUNC('month', datetime) AS month, SUM(target) AS consumption
    FROM merged
    WHERE is_consumption
    GROUP BY DATE_TRUNC('month', datetime)
),
with_lag AS (
    SELECT
        month,
        consumption,
        LAG(consumption) OVER (ORDER BY month) AS prev_consumption
    FROM monthly_cons
)

SELECT
    month,
    consumption,
    consumption - prev_consumption AS "MoM_diff",
    ROUND((((consumption - prev_consumption) / prev_consumption) * 100)::numeric, 2) AS "MoM_diff_pct"
FROM with_lag;
""",
    engine,
)

,month,consumption,MoM_diff,MoM_diff_pct
0,2021-09-01,1.610195e+07,NaN,NaN
1,2021-10-01,1.870244e+07,2600487.073,16.15
2,2021-11-01,2.100139e+07,2298945.443,12.29
3,2021-12-01,2.375600e+07,2754611.994,13.12
4,2022-01-01,2.515833e+07,1402326.673,5.90
5,2022-02-01,2.300982e+07,-2148501.174,-8.54
6,2022-03-01,2.173452e+07,-1275302.156,-5.54
7,2022-04-01,1.879101e+07,-2943515.834,-13.54
8,2022-05-01,1.769267e+07,-1098333.105,-5.84
9,2022-06-01,1.478443e+07,-2908244.248,-16.44


Consumption follows a clear seasonal pattern: peaks in winter (Dec-Jan) and drops in summer (Jun), consistent with heating demand in Estonia. The largest MoM drop is April 2023 (-29%), the largest growth is August 2022 (+20%).

## Top-5 Aggregated Consumption Values for the Last Train Month by County and Business Categories

In [ ]:
pd.read_sql(
    """
SELECT
    county
    , is_business
    , SUM(target) AS consumption
FROM merged
WHERE is_consumption
    AND datetime >= '2023-03-01'
    AND datetime < '2023-04-01'
GROUP BY county, is_business
ORDER BY consumption DESC
LIMIT 5;
""",
    engine,
)

,county,is_business,consumption
0,0,True,8834237.897
1,11,True,5145543.602
2,0,False,2168788.110
3,7,True,1610044.654
4,14,True,1498523.895


Top-5 is dominated by business segment (4 out of 5 entries). County 0 (Harjumaa) leads in both business and non-business consumption, confirming it as the most energy-intensive region, consistent with the overall aggregation results.

## Consumption Moving Averages for Business Segment in the Most Consuming County

In [ ]:
pd.read_sql(
    """
WITH daily_cons AS (
    SELECT DATE(datetime) AS date, SUM(TARGET) AS consumption
    FROM merged
    WHERE is_consumption
        AND county = 0
        AND is_business
    GROUP BY DATE(datetime)
),

with_ma AS (
    SELECT
        date,
        consumption,
        AVG(consumption) OVER (
            ORDER BY date
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ) AS "consumption_MA_3d",
        AVG(consumption) OVER (
            ORDER BY date
            ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
        ) AS "consumption_MA_7d"
    FROM daily_cons
)

SELECT date, consumption, "consumption_MA_3d", "consumption_MA_7d"
FROM with_ma
WHERE date >= '2023-03-01' AND date < '2023-04-01';
""",
    engine,
)

,date,consumption,consumption_MA_3d,consumption_MA_7d
0,2023-03-01,298244.252,309687.471000,292682.149143
1,2023-03-02,282776.647,297363.809000,284309.317143
2,2023-03-03,275194.675,285405.191333,286281.137143
3,2023-03-04,262758.441,273576.587667,286807.756000
4,2023-03-05,262927.752,266960.289333,287531.418286
5,2023-03-06,346874.150,290853.447667,291406.635000
6,2023-03-07,349790.857,319864.253000,296938.110571
7,2023-03-08,354980.548,350548.518333,305043.295714
8,2023-03-09,346247.351,350339.585333,314110.539143
9,2023-03-10,335673.140,345633.679667,322750.319857


Weekdays (Mon-Fri) show higher consumption than weekends, for example, March 11 (Sat) shows a notable drop to 278MWh vs weekday levels around 335-354MWh. MA_3d reacts faster to short-term changes, while MA_7d captures the broader weekly trend and smooths out daily fluctuations visible in raw consumption.